# 22 — Fock-Resolved Experiments

## Purpose

Probe the qubit dynamics conditioned on the storage cavity photon number. These experiments reveal the Fock-state-dependent Hamiltonian structure (dispersive shift, Stark shift, dephasing) that governs cavity–transmon interactions.

## Methodology

1. **Fock-resolved Ramsey** — perform Ramsey interferometry on the qubit conditioned on the cavity being in Fock state |n⟩; extract the n-dependent qubit frequency and dephasing rate
2. **Fock-resolved Power Rabi** — drive the qubit with variable amplitude conditioned on cavity Fock state |n⟩; observe n-dependent Rabi frequency (useful for selective gate calibration)
3. **Cavity-conditioned $T_2$ Ramsey** — measure qubit $T_2^*$ as a function of cavity photon number to quantify photon-number-dependent dephasing

## Expected Outcomes

- Ramsey fringe frequencies shifting by $2\chi$ per cavity photon
- Rabi oscillation frequencies showing $\sqrt{n}$-dependent Stark shifts at higher Fock states
- Qubit $T_2^*$ decreasing with increasing cavity photon number (photon shot-noise dephasing)

## Prerequisites

- **Notebook 21** — storage cavity characterized (frequency, χ, Fock-resolved spectroscopy)
- **Notebook 08** — displacement pulses registered for cavity state preparation

## 1. Setup — Session Bootstrap

Open the notebook stage and load the storage cavity characterization checkpoint from notebook 21.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    FockResolvedPowerRabi,
    FockResolvedRamsey,
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="22_fock_resolved_experiments",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

storage_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="21_storage_cavity_characterization",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if storage_checkpoint is not None:
    print(f"Prior stage 21 status: {storage_checkpoint['status']}")

## 2. Configuration — Fock-Resolved Experiment Defaults

Set the maximum Fock state cutoff and averaging counts for each Fock-resolved experiment.

In [ ]:
FOCK_RAMSEY_N_AVG = 10000
FOCK_RAMSEY_MAX_N = 5

FOCK_RABI_N_AVG = 10000
FOCK_RABI_MAX_N = 5

CAVITY_T2_N_AVG = 10000

print("Fock-resolved experiment settings:")
print(f"  Ramsey: max_n={FOCK_RAMSEY_MAX_N}, n_avg={FOCK_RAMSEY_N_AVG}")
print(f"  Power Rabi: max_n={FOCK_RABI_MAX_N}, n_avg={FOCK_RABI_N_AVG}")
print(f"  Cavity T2 Ramsey n_avg: {CAVITY_T2_N_AVG}")

## 3. Execution — Fock-Resolved Ramsey

Perform Ramsey interferometry on the qubit conditioned on the cavity being in Fock state |n⟩. Each Fock manifold produces a distinct Ramsey frequency shifted by multiples of $2\chi$.

In [ ]:
fock_ramsey = FockResolvedRamsey(session)
fock_ramsey_result = fock_ramsey.run(
    max_n=FOCK_RAMSEY_MAX_N,
    n_avg=FOCK_RAMSEY_N_AVG,
)
fock_ramsey_analysis = fock_ramsey.analyze(fock_ramsey_result, update_calibration=False)
fock_ramsey.plot(fock_ramsey_analysis)

print("Fock-resolved Ramsey complete.")
if hasattr(fock_ramsey_analysis, "metrics"):
    for k, v in fock_ramsey_analysis.metrics.items():
        print(f"  {k}: {v}")

## 4. Execution — Fock-Resolved Power Rabi

Drive the qubit with variable amplitude conditioned on the cavity Fock state |n⟩. Observe n-dependent Rabi frequencies, which inform selective gate pulse calibrations.

In [ ]:
fock_rabi = FockResolvedPowerRabi(session)
fock_rabi_result = fock_rabi.run(
    max_n=FOCK_RABI_MAX_N,
    n_avg=FOCK_RABI_N_AVG,
)
fock_rabi_analysis = fock_rabi.analyze(fock_rabi_result, update_calibration=False)
fock_rabi.plot(fock_rabi_analysis)

print("Fock-resolved Power Rabi complete.")
if hasattr(fock_rabi_analysis, "metrics"):
    for k, v in fock_rabi_analysis.metrics.items():
        print(f"  {k}: {v}")

## 5. Execution — Cavity-Conditioned $T_2$ Ramsey

Measure qubit $T_2^*$ as a function of cavity photon number. Increasing photon population introduces shot-noise dephasing that reduces coherence time.

In [ ]:
# This uses a displaced Fock-resolved Ramsey to extract cavity-dependent T2
cavity_t2_result = session.cavity_conditional_t2_ramsey(
    n_avg=CAVITY_T2_N_AVG,
)

print("Cavity-conditioned T2 Ramsey complete.")

## 6. Checkpoint — Save Fock-Resolved Experiment Results

Record all Fock-resolved characterization results. No calibrations are applied — these are diagnostic experiments that validate the cavity–transmon interaction model.

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="22_fock_resolved_experiments",
    status="characterized",
    summary="Fock-resolved Ramsey, Power Rabi, and cavity-conditioned T2 characterized.",
    consumed_inputs={
        "fock_ramsey_max_n": FOCK_RAMSEY_MAX_N,
        "fock_ramsey_n_avg": FOCK_RAMSEY_N_AVG,
        "fock_rabi_max_n": FOCK_RABI_MAX_N,
        "fock_rabi_n_avg": FOCK_RABI_N_AVG,
        "cavity_t2_n_avg": CAVITY_T2_N_AVG,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="23_quantum_state_preparation",
    notes=[
        "All Fock-resolved experiments are characterization-only.",
        "Cavity-conditioned T2 (Exp 67) uses session.cavity_conditional_t2_ramsey.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")